### CNN 및 DROPOUT

## STL-10 Dataset
: 자연 및 인공 사물 이미지를 10개 클래스로 분류하는 이미지 데이터셋

- 학습데이터 : 5,000장 (라벨 데이터)
- 테스트 데이터 : 8,000장
- 비라벨 학습데이터 : 100,000장
- 원본 이미지 크기 : 96 × 96 (컬러 이미지)
- 입력 데이터: RGB 이미지 (64 × 64로 크기 변경)
- 입력 특징 수: 3 × 64 × 64 = 3,072
- 출력 클래스: 10개
- 다운로드 : `split="train"`과 `split="test"`를 구분하여 다운로드
- 모델은 CNN, CNN_deep, CNN_drop (p=0.3), CNN_deep_drop (p=0.3) 4가지를 정의하여 사용

| 클래스 이름 | 의미 |
|:---:|:---|
| `airplane` | 비행기 |
| `bird` | 새 |
| `car` | 자동차 |
| `cat` | 고양이 |
| `deer` | 사슴 |
| `dog` | 개 |
| `horse` | 말 |
| `monkey` | 원숭이 |
| `ship` | 선박 |
| `truck` | 트럭 |

LR = 1e-3<br>
EPOCH = 30<br>
**test_DS.class_to_idx**는 기본제공되지 않음

1. 문제 1
- train, test데이터 모두 transforms는 사이즈 64 x 64 적용

2. 문제 2
- train 데이터의 transforms는
    - 사이즈 64 x 64 적용
    - 이미지의 좌우 반전을 50%의 확률로 적용
    - 입력 이미지의 상하좌우에 4픽셀씩 패딩을 추가한 후, 64×64 크기를 무작위로 잘라내기 (transforms.RandomCrop(64, padding=4))
    - Normalize((0.4467, 0.4398, 0.4066),(0.2603, 0.2566, 0.2713)) 적용

- test 데이터의 transforms는
    - 사이즈 64 x 64 적용
    - Normalize((0.4467, 0.4398, 0.4066),(0.2603, 0.2566, 0.2713)) 적용

**실행시 문제1과 문제2는 하나씩 선택하여 실행**

In [3]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

PROBLEM=1
LR=1e-3
EPOCH=10
BATCH_SIZE=64
SEED=42

classes=["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)

mean=(0.4467, 0.4398, 0.4066)
std=(0.2603, 0.2566, 0.2713)

if PROBLEM == 1:
    train_transform=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        ])
    test_transfrom=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        ])
elif PROBLEM==2:
    train_transform=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomCrop(64, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_transfrom=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
else:
    raise ValueError("PROBLEM은 1 또는 2만 선택")

train_DS=datasets.STL10(root="./data", split="train", download=True, transform=train_transform)
test_DS=datasets.STL10(root="./data", split="test", download=True, transform=test_transfrom)

train_loader=DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader=DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

cpu


100.0%


In [13]:
# class CNN(nn.Module):
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.features=nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),
#         )
#         self.classifier=nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(64*16*16, 256),
#             nn.ReLU(inplace=True),
#             nn.Linear(256, num_classes),
#         )

#     def forward(self, x):
#         x=self.features(x)
#         return x

# class CNN_deep(nn.Module):
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.features=nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),

#             nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2)
#         )
#         self.classifier=nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(128*8*8, 256),
#             nn.ReLU(inplace=True),
#             nn.Linear(256, num_classes),
#         )

#     def forward(self, x):
#         x=self.features(x)
#         return x

# class CNN_drop(nn.Module):
#     def __init__(self, p=0.3, num_classes=10):
#         super().__init__()
#         self.features=nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),
#         )
#         self.classifier=nn.Sequential(
#             nn.Flatten(),
#             nn.Dropout(p=p),
#             nn.Linear(64*16*16, 256),
#             nn.ReLU(inplace=True),
#             nn.Dropout(p=p),
#             nn.Linear(256, num_classes),   
#         )

#     def forward(self, x):
#         x=self.features(x)
#         return x

# class CNN_deep_drop(nn.Module):
#     def __init__(self, p=0.3, num_classes=10):
#         super().__init__()
#         self.features=nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
#             nn.MaxPool2d(2),

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),

#             nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),
#         )
#         self.classifier=nn.Sequential(
#             nn.Flatten(),
#             nn.Dropout(p=p),
#             nn.Linear(128*8*8, 256),
#             nn.ReLU(inplace=True),
#             nn.Dropout(p=p),
#             nn.Linear(256, num_classes),
#         )

#     def forward(self, x):
#         x=self.features(x)
#         return x

# def get_models():
#     return {
#         "CNN": CNN().to(device),
#         "CNN_deep": CNN_deep().to(device),
#         "CNN_drop": CNN_drop(p=0.3).to(device),
#         "CNN_deep_drop": CNN_deep_drop(p=0.3).to(device),
#     }
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class CNN_deep(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class CNN_drop(nn.Module):
    def __init__(self, p=0.3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=p),
            nn.Linear(64 * 16 * 16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class CNN_deep_drop(nn.Module):
    def __init__(self, p=0.3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=p),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=p),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_models():
    return {
        "CNN": CNN().to(device),
        "CNN_deep": CNN_deep().to(device),
        "CNN_drop": CNN_drop(p=0.3).to(device),
        "CNN_deep_drop": CNN_deep_drop(p=0.3).to(device),
    }

In [14]:
# @torch.no_grad()
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total= 0.0, 0, 0
    for images, labels in loader:
        images, labels=images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss+=loss.item() *images.size(0)
        preds=outputs.argmax(dim=1)
        correct+=(preds==labels).sum().item()
        total+=labels.size(0)
    return running_loss / total, 100.0 * correct / total

In [15]:
criterion=nn.CrossEntropyLoss()
models=get_models()
results={}

for name, model in models.items():
    set_seed(SEED)
    optimizer=optim.Adam(model.parameters(), lr=LR)
    print(f' 학습 시작 ')
    for epoch in range(1, EPOCH+1):
        train_loss, train_acc=train_one_epoch(model, train_loader, criterion, optimizer)
        test_loss, test_acc=evaluate(model, test_loader, criterion)
        print(
            f'Epoch [{epoch:02d}/{EPOCH:02d}]'
            f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | '
            f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%'
        )
    results[name]=test_acc

print('최종')
for name, acc in results.items():
    print(f'{name:15s}: {acc:.2f}%')

 학습 시작 


KeyboardInterrupt: 